In [6]:
import numpy as np
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
DATA_DIR = Path("../data_csv")

P_PATH   = DATA_DIR / "persona_vectors.npy"
X_PATH   = DATA_DIR / "final_product_with_brandtone.npy"
META_PATH= DATA_DIR / "final_product_with_brandtone_meta.csv"

OUT_FULL = DATA_DIR / "persona_product_similarity_full.csv"
OUT_TOPN = DATA_DIR / "persona_product_similarity_topN.csv"
OUT_STR  = DATA_DIR / "persona_brand_tone_strategy.csv"

TOPN = 50

# ------------------------------------------------------------
# 1) LOAD
# ------------------------------------------------------------
P = np.load(P_PATH).astype(np.float32)   # (P, Dp)
X = np.load(X_PATH).astype(np.float32)   # (N, Dx)
meta = pd.read_csv(META_PATH).reset_index(drop=True)

assert X.shape[0] == len(meta), "❌ 제품 벡터 수와 메타 행 수 불일치"

Dp = P.shape[1]
Dx = X.shape[1]

# ------------------------------------------------------------
# 🔧 차원 정렬 (핵심)
# ------------------------------------------------------------
if Dp > Dx:
    print(f"⚠️ persona dim({Dp}) > product dim({Dx}) → persona slicing")
    P = P[:, :Dx]

elif Dp < Dx:
    print(f"⚠️ persona dim({Dp}) < product dim({Dx}) → persona padding with zeros")
    pad = Dx - Dp
    P = np.pad(P, ((0, 0), (0, pad)), mode="constant")

print(f"✅ embedding dim aligned: {P.shape[1]}")

# ------------------------------------------------------------
# 2) NORMALIZE (cosine similarity)
# ------------------------------------------------------------
P_norm = P / (np.linalg.norm(P, axis=1, keepdims=True) + 1e-12)
X_norm = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)

# ------------------------------------------------------------
# 3) SIMILARITY (P × N)
# ------------------------------------------------------------
S = P_norm @ X_norm.T   # (P, N)

# ------------------------------------------------------------
# 4) FULL SAVE
# ------------------------------------------------------------
persona_ids = [f"persona_{i}" for i in range(1, P.shape[0] + 1)]

brand_col = (
    "brand" if "brand" in meta.columns
    else "브랜드" if "브랜드" in meta.columns
    else None
)

full_rows = []
for pi, pid in enumerate(persona_ids):
    for j, sim in enumerate(S[pi]):
        row = {
            "persona_id": pid,
            "product_index": j,
            "similarity": float(sim),
        }
        if brand_col:
            row["brand"] = meta.loc[j, brand_col]
        full_rows.append(row)

df_full = pd.DataFrame(full_rows)
df_full.to_csv(OUT_FULL, index=False)
print("✅ saved:", OUT_FULL, "rows:", len(df_full))

# ------------------------------------------------------------
# 5) TOP-N SAVE
# ------------------------------------------------------------
top_rows = []
for pi, pid in enumerate(persona_ids):
    sims = S[pi]
    top_idx = np.argsort(-sims)[:TOPN]
    for rank, j in enumerate(top_idx, start=1):
        row = {
            "persona_id": pid,
            "rank": rank,
            "product_index": j,
            "similarity": float(sims[j]),
        }
        if brand_col:
            row["brand"] = meta.loc[j, brand_col]
        top_rows.append(row)

df_top = pd.DataFrame(top_rows)
df_top.to_csv(OUT_TOPN, index=False)
print("✅ saved:", OUT_TOPN, "rows:", len(df_top))

# ------------------------------------------------------------
# 6) persona × brand × brand_tone_cluster
# ------------------------------------------------------------
SEG_PATH = DATA_DIR / "final_brand_segments.csv"
seg = pd.read_csv(SEG_PATH)

if "brand" not in seg.columns and "브랜드" in seg.columns:
    seg = seg.rename(columns={"브랜드": "brand"})

required = {"brand", "brand_tone_cluster"}
missing = required - set(seg.columns)
if missing:
    raise ValueError(f"❌ final_brand_segments.csv missing: {missing}")

merged = df_full.merge(
    seg[["brand", "brand_tone_cluster"]],
    on="brand",
    how="left"
)

persona_brand = (
    merged
    .groupby(["persona_id", "brand", "brand_tone_cluster"], dropna=False)
    .agg(
        avg_similarity=("similarity", "mean"),
        product_cnt=("product_index", "count")
    )
    .reset_index()
)

persona_brand.to_csv(OUT_STR, index=False)
print("✅ saved:", OUT_STR, "rows:", len(persona_brand))

⚠️ persona dim(1540) < product dim(2560) → persona padding with zeros
✅ embedding dim aligned: 2560
✅ saved: ../data_csv/persona_product_similarity_full.csv rows: 12648
✅ saved: ../data_csv/persona_product_similarity_topN.csv rows: 400
✅ saved: ../data_csv/persona_brand_tone_strategy.csv rows: 400
